# 08 · Real energies — the tier ladder

Embedding, search and prune are geometry-driven (pure RDKit + openconf) — no calculator needed. To *rank* conformers you need a real energy: the FF is meaningless for a metal or charged TS. The ladder is **`ff → gfnff → gfn2 → gxtb`** (force field → GFN-FF, dispersion + H-bond, ~FF cost → GFN2 → g-xTB). `score()` is a single point (no geometry change); `optimize()` relaxes (holding any frozen core). Energies are kcal/mol; only *relative* energies are meaningful. Needs an `xtb` binary (here `~/bin/xtb`).

In [ ]:
import tempfile

import rxembed as rx
import xyzrender

rx.set_verbose("INFO")
IBU = "CC(C)Cc1ccc(C(C)C(=O)O)cc1"  # ibuprofen — several real rotamers

## The pool — embed → search → prune → keep the lowest few

Build a diverse, deduplicated pool on the cheap FF, then spend the calculator only on the best candidates.

In [ ]:
pool = rx.embed(IBU).mc().prune()
cand = pool.lowest(6)  # the 6 lowest on the force field, to re-rank with a real energy
print(f"{len(pool)} pruned conformers -> re-ranking the lowest {len(cand)}")

## Real single points — g-xTB

`score()` re-ranks on an honest single-point energy without moving atoms. `relative()` gives ΔE from the minimum — and the FF order is **not** the g-xTB order, which is the whole point of scoring.

In [ ]:
scored = cand.score("gxtb")
de = sorted(scored.relative().values())
print("g-xTB ΔE (kcal/mol):", [round(e, 2) for e in de])
print("lowest by g-xTB:", scored.lowest(1).ids)

## The recommended pipeline — GFN-FF pool re-rank, then g-xTB on the few

GFN-FF has dispersion + H-bond terms the UFF relax lacks, at ~FF cost — the right tier to rank the *whole pool* (the first NCI-aware energy in the loop), before spending g-xTB on only the survivors. One clear, not-slow call each.

In [ ]:
ranked = pool.score("gfnff").prune()  # NCI-aware pool re-rank
print("GFN-FF ΔE:", sorted(round(e, 1) for e in ranked.relative().values())[:6])
final = ranked.lowest(2).optimize("gxtb", level="loose")  # g-xTB opt, the best few only
print("g-xTB opt ΔE:", sorted(round(e, 2) for e in final.relative().values()))
xyzrender.render(final.lowest(1).dump(tempfile.mktemp(suffix=".xyz")), no_hy=True)

## Honest energies, non-clobbering

`score('gxtb')` / `optimize('gxtb')` **raise** if the calculator produces nothing — never a silent FF fallback the caller mistakes for xTB (g-xTB + solvent also raises, no ALPB). And they return **new** ensembles: the input geometry is never moved under you.

In [ ]:
print("input pool untouched:", len(pool), "conformers on the FF path")
print("scored / optimised are independent ensembles:", len(scored), "/", len(final))